In [1]:
# Montaje del Drive
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


In [2]:
%%bash
# Replicación de la estructura de carpetas
mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf" /content/buckets/b1

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

# Descarga condicional del crudo
FILE_DRIVE="/content/buckets/b1/datasets/competencia_01_crudo.csv"
FILE_LOCAL="/content/datasets/competencia_01_crudo.csv"
URL="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv"

if [ ! -f "$FILE_DRIVE" ]; then
  wget "$URL" -O "$FILE_DRIVE"
fi

if [ ! -f "$FILE_LOCAL" ]; then
  cp "$FILE_DRIVE" "$FILE_LOCAL"
fi

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 17.4 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 1. FUNCIÓN: RANKING CON CERO FIJO Y ESCALADO SIMÉTRICO
# =============================================================================
def aplicar_rank_cero_fijo(df, columnas_a_rankear):
    """
    Convierte variables numéricas a pseudo-percentiles por 'foto_mes' manteniendo el 0:
    - Positivos: Escalados proporcionalmente entre 0.001 y 1.0.
    - Negativos: Escalados proporcionalmente entre -1.0 y -0.001.
    - Ceros y NaNs: Se mantienen intactos.
    """
    df_out = df.copy()

    def escalar_rangos(s, val_min, val_max):
        if s.empty:
            return s

        # rank(method='dense') asigna rangos enteros consecutivos
        r = s.rank(method='dense')
        r_min, r_max = r.min(), r.max()

        # Si hay un único valor en la distribución del mes, se le asigna el máximo (en magnitud)
        if r_min == r_max:
            return pd.Series(val_max, index=s.index)

        # Escalado Min-Max a un rango [0, 1] y luego proyección al intervalo [val_min, val_max]
        p = (r - r_min) / (r_max - r_min)
        return val_min + p * (val_max - val_min)

    for col in columnas_a_rankear:
        # Máscaras booleanas (los NaN evalúan como False en ambas comparaciones)
        mask_pos = df_out[col] > 0
        mask_neg = df_out[col] < 0

        # Aplicar escalado a los positivos agrupados por foto_mes
        if mask_pos.any():
            df_out.loc[mask_pos, col] = (
                df_out.loc[mask_pos]
                .groupby('foto_mes')[col]
                .transform(lambda x: escalar_rangos(x, 0.001, 1.0))
            )

        # Aplicar escalado a los negativos agrupados por foto_mes
        if mask_neg.any():
            df_out.loc[mask_neg, col] = (
                df_out.loc[mask_neg]
                .groupby('foto_mes')[col]
                .transform(lambda x: escalar_rangos(x, -1.0, -0.001))
            )

    return df_out
# =============================================================================
# 2. CARGA Y CREACIÓN DE LA CLASE BINARIA
# =============================================================================
print("Cargando dataset...")
dataset = pd.read_csv("/content/datasets/competencia_01_crudo.csv")

# Cálculo de meses consecutivos para ubicar bajas
dataset['periodo0'] = (dataset['foto_mes'] // 100) * 12 + (dataset['foto_mes'] % 100)
dataset.sort_values(['numero_de_cliente', 'periodo0'], inplace=True)

# Leads para mirar hacia el futuro
dataset['periodo1'] = dataset.groupby('numero_de_cliente')['periodo0'].shift(-1)
dataset['periodo2'] = dataset.groupby('numero_de_cliente')['periodo0'].shift(-2)

periodo_ultimo = dataset['periodo0'].max()
periodo_anteultimo = periodo_ultimo - 1

dataset['clase_binaria'] = np.nan
dataset.loc[dataset['periodo0'] < periodo_anteultimo, 'clase_binaria'] = 'CONTINUA'

# Lógica BAJA (Agrupa BAJA+1 y BAJA+2 en una sola etiqueta)
mask_baja1 = (dataset['periodo0'] < periodo_ultimo) & \
             (dataset['periodo1'].isna() | (dataset['periodo0'] + 1 < dataset['periodo1']))

mask_baja2 = (dataset['periodo0'] < periodo_anteultimo) & \
             (dataset['periodo0'] + 1 == dataset['periodo1']) & \
             (dataset['periodo2'].isna() | (dataset['periodo0'] + 2 < dataset['periodo2']))

dataset.loc[mask_baja1 | mask_baja2, 'clase_binaria'] = 'BAJA'

# Limpieza y Target numérico
dataset.drop(columns=['periodo0', 'periodo1', 'periodo2'], inplace=True)
dataset.dropna(subset=['clase_binaria'], inplace=True) # Excluye meses 202107/08 sin clase
dataset['target'] = np.where(dataset['clase_binaria'] == 'BAJA', 1, 0)

# =============================================================================
# 3. APLICACIÓN DE RANKING (Previo a la creación de deltas/lags)
# =============================================================================
print("Aplicando Ranking Cero Fijo a las variables base...")
features_base = [c for c in dataset.columns if c not in ['numero_de_cliente', 'foto_mes', 'clase_binaria', 'target']]

# Se rankean primero las variables originales
dataset = aplicar_rank_cero_fijo(dataset, features_base)

# =============================================================================
# 4. FEATURE ENGINEERING: LAGS Y DELTAS (Sobre variables rankeadas)
# =============================================================================
print("Calculando Lags y Deltas...")
# Garantizar el orden estrictamente temporal
dataset.sort_values(['numero_de_cliente', 'foto_mes'], inplace=True)

# Recreamos un período continuo para validar distancias exactas de meses
dataset['periodo_temp'] = (dataset['foto_mes'] // 100) * 12 + (dataset['foto_mes'] % 100)

for col in features_base:
    # Lags crudos usando el desplazamiento de filas por cliente
    dataset[f'{col}_LAG1'] = dataset.groupby('numero_de_cliente')[col].shift(1)
    dataset[f'{col}_LAG2'] = dataset.groupby('numero_de_cliente')[col].shift(2)

    # Desplazamiento del período para verificar la temporalidad estricta
    periodo_lag1 = dataset.groupby('numero_de_cliente')['periodo_temp'].shift(1)
    periodo_lag2 = dataset.groupby('numero_de_cliente')['periodo_temp'].shift(2)

    # Si la fila anterior no es exactamente el mes pasado (diferencia != 1), forzar NaN
    dataset.loc[dataset['periodo_temp'] - periodo_lag1 != 1, f'{col}_LAG1'] = np.nan

    # Si dos filas atrás no son exactamente hace dos meses (diferencia != 2), forzar NaN
    dataset.loc[dataset['periodo_temp'] - periodo_lag2 != 2, f'{col}_LAG2'] = np.nan

    # Deltas: Si alguno de los componentes es NaN, pandas devuelve NaN automáticamente
    dataset[f'{col}_DELTA-LAG1'] = dataset[col] - dataset[f'{col}_LAG1']
    dataset[f'{col}_DELTA-LAG2'] = dataset[col] - dataset[f'{col}_LAG2']

# Limpieza de la columna temporal de soporte
dataset.drop(columns=['periodo_temp'], inplace=True)

# Actualizamos las features_totales para incluir las 4 nuevas variables por columna
features_totales = [c for c in dataset.columns if c not in ['numero_de_cliente', 'foto_mes', 'clase_binaria', 'target']]
# =============================================================================
# 5. SPLIT TEMPORAL (Train: 202103/04 - Test: 202106)
# =============================================================================
train_mask = dataset['foto_mes'].isin([202103, 202104])
X_train = dataset.loc[train_mask, features_totales]
y_train = dataset.loc[train_mask, 'target']

test_mask = dataset['foto_mes'] == 202106
X_test = dataset.loc[test_mask, features_totales]
y_test = dataset.loc[test_mask, 'target']

print(f"Dimensiones Train: {X_train.shape} | Dimensiones Test: {X_test.shape}")

# =============================================================================
# 6. OPTIMIZACIÓN BAYESIANA CON OPTUNA (Algoritmo TPE para LightGBM)
# =============================================================================
def objective(trial):
    # Espacio de búsqueda de hiperparámetros (LightGBM)
    params = {
        # --- 1. Hiperparámetros a optimizar (Búsqueda Logarítmica) ---
        'n_estimators': trial.suggest_int('n_estimators', 64, 4096, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 2048, log=True),
        'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-06, 0.1, log=True),

        # --- 2. Parámetros fijos ---
        'learning_rate': 0.005,
        'min_child_samples': 0,        # Equivale a min_data_in_leaf (CERO fijo)
        'colsample_bytree': 0.50,      # Equivale a feature_fraction
        'max_depth': -1,
        'min_split_gain': 0.0,         # Equivale a min_gain_to_split
        'reg_alpha': 0.0,              # Equivale a lambda_l1
        'reg_lambda': 0.0,             # Equivale a lambda_l2
        'max_bin': 31,
        'subsample': 1.0,              # Equivale a bagging_fraction
        'is_unbalance': False,
        'scale_pos_weight': 1.0,
        'extra_trees': False,
        'boosting_type': 'gbdt',
        'objective': 'binary',
        'feature_pre_filter': False,
        'deterministic': True,
        'verbosity': -100,
        'random_state': 331613,        # Semilla actualizada
        'n_jobs': -1
    }

    # Instanciamos LightGBM
    clf = LGBMClassifier(**params)
    clf.fit(X_train, y_train)

    # Predecir probabilidades de BAJA (clase 1) en el mes de validación
    y_pred_proba = clf.predict_proba(X_test)[:, 1]

    # Métrica a maximizar: Área bajo la curva ROC
    auc = roc_auc_score(y_test, y_pred_proba)

    return auc

print("Iniciando búsqueda bayesiana de hiperparámetros...")

# Implementación del sampler TPE configurado con 10 trials de startup y la nueva semilla
sampler = optuna.samplers.TPESampler(n_startup_trials=10, seed=331613)

study = optuna.create_study(
    direction='maximize',
    study_name="HT_LightGBM_Optuna",
    sampler=sampler
)
study.optimize(objective, n_trials=20)

print("\n================ RESULTADOS ================")
print(f"Mejor AUC: {study.best_value}")
print("Mejores hiperparámetros identificados:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Cargando dataset...
Aplicando Ranking Cero Fijo a las variables base...
Calculando Lags y Deltas...


[I 2026-10-11 02:48:01,171] A new study created in memory with name: HT_LightGBM_Optuna


Dimensiones Train: (326184, 760) | Dimensiones Test: (164114, 760)
Iniciando búsqueda bayesiana de hiperparámetros...


[I 2026-10-11 02:53:04,767] Trial 0 finished with value: 0.9094891168317816 and parameters: {'n_estimators': 349, 'num_leaves': 115, 'min_sum_hessian_in_leaf': 6.232678945573573e-06}. Best is trial 0 with value: 0.9094891168317816.
[I 2026-10-11 02:56:15,826] Trial 1 finished with value: 0.9036695068928013 and parameters: {'n_estimators': 228, 'num_leaves': 27, 'min_sum_hessian_in_leaf': 9.770939208343195e-05}. Best is trial 0 with value: 0.9094891168317816.
[I 2026-10-11 03:04:54,681] Trial 2 finished with value: 0.9080310864543318 and parameters: {'n_estimators': 707, 'num_leaves': 22, 'min_sum_hessian_in_leaf': 0.00012647019083385902}. Best is trial 0 with value: 0.9094891168317816.
[I 2026-10-11 03:08:58,072] Trial 3 finished with value: 0.9099205963580416 and parameters: {'n_estimators': 259, 'num_leaves': 216, 'min_sum_hessian_in_leaf': 0.07382622568202596}. Best is trial 3 with value: 0.9099205963580416.
[I 2026-10-11 03:12:43,154] Trial 4 finished with value: 0.9083184522629266


================ RESULTADOS ================
Mejor AUC: 0.9144283303321411
Mejores hiperparámetros identificados:
  n_estimators: 1273
  num_leaves: 721
  min_sum_hessian_in_leaf: 0.003055877921643489
